## Installing voice dependencies

In [1]:
from pathlib import Path
import re
import subprocess
import sys

# Locate the project root.
search_paths = [Path.cwd(), *Path.cwd().parents]

PROJECT_DIR = next(
    (
        path for path in search_paths
        if (path / "requirements.txt").is_file()
    ),
    None
)

if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Could not find requirements.txt. "
        "Open this notebook inside the project directory."
    )

requirements_path = PROJECT_DIR / "requirements.txt"
existing_text = requirements_path.read_text(encoding="utf-8")

# Add only packages not already registered.
existing_names = set()

for line in existing_text.splitlines():
    line = line.strip()

    if not line or line.startswith(("#", "-")):
        continue

    match = re.match(r"([A-Za-z0-9_.-]+)", line)

    if match:
        existing_names.add(
            match.group(1).lower().replace("_", "-")
        )

voice_packages = ["groq", "edge-tts", "streamlit"]

missing_packages = [
    package for package in voice_packages
    if package not in existing_names
]

if missing_packages:
    requirements_path.write_text(
        existing_text.rstrip()
        + "\n"
        + "\n".join(missing_packages)
        + "\n",
        encoding="utf-8"
    )

print("Project root:", PROJECT_DIR)
print("Requirements updated:", requirements_path)
print("New dependencies:", missing_packages or "Already registered")

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    *voice_packages
])

print("\nVoice dependencies installed successfully.")

Project root: c:\Hehe\AI-Lead-Desk-Runtime-Errors
Requirements updated: c:\Hehe\AI-Lead-Desk-Runtime-Errors\requirements.txt
New dependencies: Already registered

Voice dependencies installed successfully.


In [2]:
# Cell 3: Install and register ipywidgets

from pathlib import Path
import re
import subprocess
import sys

# Reuse PROJECT_DIR if it exists; otherwise locate the project root.
if "PROJECT_DIR" not in globals() or not PROJECT_DIR.exists():
    search_paths = [Path.cwd(), *Path.cwd().parents]

    PROJECT_DIR = next(
        (
            path for path in search_paths
            if (path / "requirements.txt").is_file()
        ),
        None
    )

if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Could not locate the project requirements.txt."
    )

requirements_path = PROJECT_DIR / "requirements.txt"
existing_text = requirements_path.read_text(encoding="utf-8")

registered_names = set()

for line in existing_text.splitlines():
    line = line.strip()

    if not line or line.startswith(("#", "-")):
        continue

    match = re.match(r"([A-Za-z0-9_.-]+)", line)

    if match:
        registered_names.add(
            match.group(1).lower().replace("_", "-")
        )

if "ipywidgets" not in registered_names:
    requirements_path.write_text(
        existing_text.rstrip() + "\nipywidgets\n",
        encoding="utf-8"
    )
    print("Added ipywidgets to requirements.txt.")
else:
    print("ipywidgets is already registered.")

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "ipywidgets"
])

import ipywidgets

print("\nipywidgets installed successfully.")
print("Version:", ipywidgets.__version__)
print("Python environment:", sys.executable)

ipywidgets is already registered.

ipywidgets installed successfully.
Version: 8.1.9
Python environment: D:\Anaconda\envs\pytorch\python.exe


In [5]:
# Cell 1: Initialize Gemini API client

import os
from pathlib import Path
from dotenv import load_dotenv
from google import genai

# Locate the project root.
PROJECT_DIR = Path(r"C:\Hehe\AI-Lead-Desk-Runtime-Errors")

if not (PROJECT_DIR / ".env").is_file():
    raise FileNotFoundError(
        f"Project .env file not found: {PROJECT_DIR / '.env'}"
    )

load_dotenv(PROJECT_DIR / ".env", override=True)

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise EnvironmentError(
        "GEMINI_API_KEY is missing from the project's .env file."
    )

gemini_client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini client initialized successfully.")
print("API key loaded:", bool(GEMINI_API_KEY))
print("Project:", PROJECT_DIR)

Gemini client initialized successfully.
API key loaded: True
Project: C:\Hehe\AI-Lead-Desk-Runtime-Errors


In [6]:
# Diagnose Gemini transcription latency

import time
from pathlib import Path

VOICE_SAMPLE_DIR = PROJECT_DIR / "data" / "voice_samples"

audio_files = sorted(
    (
        p for p in VOICE_SAMPLE_DIR.iterdir()
        if p.is_file()
        and p.suffix.lower() in {
            ".wav", ".mp3", ".m4a", ".ogg",
            ".flac", ".mp4", ".webm"
        }
    ),
    key=lambda p: p.stat().st_mtime,
    reverse=True
)

if not audio_files:
    raise FileNotFoundError("No audio recordings found.")

audio_path = audio_files[0]
file_size_mb = audio_path.stat().st_size / (1024 * 1024)

print(f"Recording: {audio_path.name}", flush=True)
print(f"File size: {file_size_mb:.2f} MB", flush=True)

# Stage 1: Upload the audio.
print("\n[1/2] Uploading audio to Google...", flush=True)
upload_start = time.perf_counter()

uploaded_audio = gemini_client.files.upload(
    file=str(audio_path)
)

upload_seconds = time.perf_counter() - upload_start
print(
    f"Upload completed in {upload_seconds:.2f} seconds.",
    flush=True
)

# Stage 2: Transcribe with automatic language detection.
print("\n[2/2] Transcribing with Gemini...", flush=True)
transcription_start = time.perf_counter()

interaction = gemini_client.interactions.create(
    model="gemini-3.5-transcribe",
    input=[
        {
            "type": "audio",
            "uri": uploaded_audio.uri,
            "mime_type": uploaded_audio.mime_type,
        }
    ],
    generation_config={
        "transcription_config": {
            "language_codes": [],
            "custom_vocabulary": [
                "Python Programming",
                "Data Science Foundations",
                "Web Development",
                "AI Lead Desk",
            ],
        }
    },
)

transcription_seconds = time.perf_counter() - transcription_start
transcript = (interaction.output_text or "").strip()

print(
    f"Transcription completed in {transcription_seconds:.2f} seconds.",
    flush=True
)
print(f"Total time: {upload_seconds + transcription_seconds:.2f} seconds")

print("\nTRANSCRIPT:")
print(transcript or "[No transcript returned]")

Recording: Recording (20).m4a
File size: 0.09 MB

[1/2] Uploading audio to Google...
Upload completed in 4.49 seconds.

[2/2] Transcribing with Gemini...
Transcription completed in 5.98 seconds.
Total time: 10.47 seconds

TRANSCRIPT:
Python Programming की प्राइस क्या है?


In [10]:
# Test Gemini 3.8 Flash-Lite TTS in Hindi and Marathi

import os
import base64
from pathlib import Path
from dotenv import load_dotenv
from google import genai
from IPython.display import Audio, display

# Reuse the existing Gemini client or initialize it if necessary.
load_dotenv(PROJECT_DIR / ".env")

if "gemini_client" not in globals():
    api_key = os.getenv("GEMINI_API_KEY")
    if not api_key:
        raise EnvironmentError("GEMINI_API_KEY is missing from .env")
    gemini_client = genai.Client(api_key=api_key)

TTS_MODEL = "gemini-3.8-flash-lite-tts"

TTS_TESTS = {
    "hindi": "नमस्ते! Python Programming कोर्स की फीस आठ हजार रुपये है।",
    "marathi": "नमस्कार! Python Programming कोर्सची फी आठ हजार रुपये आहे।",
}

TTS_OUTPUT_DIR = PROJECT_DIR / "data" / "voice_outputs"
TTS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

generated_audio = {}

for language, text in TTS_TESTS.items():
    output_path = TTS_OUTPUT_DIR / f"gemini_tts_{language}.wav"

    print(f"\nGenerating {language.title()} speech...", flush=True)

    interaction = gemini_client.interactions.create(
        model=TTS_MODEL,
        input=[
            {
                "type": "user_input",
                "content": [
                    {
                        "type": "text",
                        "text": text,
                        "annotations": [
                            {
                                "type": "speech_metadata",
                                "style": "Speak naturally and clearly at a conversational pace."
                            }
                        ],
                    }
                ],
            }
        ],
        response_format={"type": "audio"},
        generation_config={
            "speech_config": [
                {"voice": "Kore"}
            ]
        },
    )

    audio_data = interaction.output_audio.data

    if not audio_data:
        raise RuntimeError(
            f"No audio was returned for {language}."
        )

    output_path.write_bytes(base64.b64decode(audio_data))

    if output_path.stat().st_size == 0:
        raise RuntimeError(f"Empty audio file for {language}.")

    generated_audio[language] = output_path

    print("Saved:", output_path)
    print("Size:", f"{output_path.stat().st_size:,} bytes")
    print(f"\nListen to {language.title()}:")

    display(Audio(filename=str(output_path), autoplay=False))

print("\nBoth language tests completed.")


Generating Hindi speech...
Saved: C:\Hehe\AI-Lead-Desk-Runtime-Errors\data\voice_outputs\gemini_tts_hindi.wav
Size: 236,514 bytes

Listen to Hindi:



Generating Marathi speech...
Saved: C:\Hehe\AI-Lead-Desk-Runtime-Errors\data\voice_outputs\gemini_tts_marathi.wav
Size: 236,514 bytes

Listen to Marathi:



Both language tests completed.


In [11]:
# Verify that Hindi and Marathi TTS produced distinct audio files

import hashlib
import wave
from pathlib import Path

TTS_OUTPUT_DIR = PROJECT_DIR / "data" / "voice_outputs"

audio_files = {
    "Hindi": TTS_OUTPUT_DIR / "gemini_tts_hindi.wav",
    "Marathi": TTS_OUTPUT_DIR / "gemini_tts_marathi.wav",
}

file_hashes = {}

for language, path in audio_files.items():
    if not path.is_file():
        print(f"{language}: File not found")
        continue

    audio_bytes = path.read_bytes()
    file_hash = hashlib.sha256(audio_bytes).hexdigest()
    file_hashes[language] = file_hash

    print(f"\n{language}")
    print("File:", path.name)
    print("Size:", len(audio_bytes), "bytes")
    print("SHA-256:", file_hash)

    try:
        with wave.open(str(path), "rb") as audio:
            duration = audio.getnframes() / audio.getframerate()
            print("Duration:", round(duration, 2), "seconds")
            print("Sample rate:", audio.getframerate(), "Hz")
            print("Channels:", audio.getnchannels())
    except (wave.Error, EOFError):
        print("Could not read WAV metadata.")

if len(file_hashes) == 2:
    print("\nAre the audio files byte-for-byte identical?",
          file_hashes["Hindi"] == file_hashes["Marathi"])


Hindi
File: gemini_tts_hindi.wav
Size: 236514 bytes
SHA-256: 8652f970360d66ec1db0685285d97276b32ff322df3b20807cf3bb8f078a1fad
Duration: 4.8 seconds
Sample rate: 24000 Hz
Channels: 1

Marathi
File: gemini_tts_marathi.wav
Size: 236514 bytes
SHA-256: 33a98baf79b16581d5ae133337f15b1112dea6dc7e55c761bf313bd0b69cba00
Duration: 4.8 seconds
Sample rate: 24000 Hz
Channels: 1

Are the audio files byte-for-byte identical? False


In [12]:
# Cell 11: Gemini TTS as the main speech-generation function

import base64
import os
from pathlib import Path
from dotenv import load_dotenv

from google import genai

# Ensure the client is available.
load_dotenv(PROJECT_DIR / ".env")

if "gemini_client" not in globals():
    api_key = os.getenv("GEMINI_API_KEY")
    if not api_key:
        raise EnvironmentError("GEMINI_API_KEY is missing from .env")

    gemini_client = genai.Client(api_key=api_key)

GEMINI_TTS_MODEL = "gemini-3.8-flash-lite-tts"

TTS_OUTPUT_DIR = PROJECT_DIR / "data" / "voice_outputs"
TTS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LANGUAGE_INSTRUCTIONS = {
    "en": (
        "Speak the supplied text naturally in English. "
        "Do not translate or paraphrase it."
    ),
    "hi": (
        "Speak the supplied text naturally in Hindi. "
        "Do not translate or paraphrase it."
    ),
    "mr": (
        "Speak the supplied text naturally in Marathi. "
        "Do not translate or paraphrase it."
    ),
}


async def synthesize_speech(
    text,
    language=None,
    output_path=None
):
    """
    Generate spoken audio with Gemini TTS.

    language: Optional en, hi, or mr instruction.
    The spoken content should already be in the target language.
    """
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Text to speak cannot be empty.")

    if language not in (None, "en", "hi", "mr"):
        raise ValueError("language must be None, en, hi, or mr.")

    if output_path is None:
        output_path = (
            TTS_OUTPUT_DIR / "latest_counsellor_response.wav"
        )

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    style = LANGUAGE_INSTRUCTIONS.get(
        language,
        "Speak the supplied text naturally and clearly in its original language. "
        "Do not translate or paraphrase it."
    )

    response = gemini_client.interactions.create(
        model=GEMINI_TTS_MODEL,
        input=[
            {
                "type": "user_input",
                "content": [
                    {
                        "type": "text",
                        "text": text.strip(),
                        "annotations": [
                            {
                                "type": "speech_metadata",
                                "style": style
                            }
                        ],
                    }
                ],
            }
        ],
        response_format={"type": "audio"},
        generation_config={
            "speech_config": [{"voice": "Kore"}]
        },
    )

    audio_data = response.output_audio.data

    if not audio_data:
        raise RuntimeError("Gemini returned no speech audio.")

    if isinstance(audio_data, str):
        audio_bytes = base64.b64decode(audio_data)
    else:
        audio_bytes = bytes(audio_data)

    output_path.write_bytes(audio_bytes)

    if not output_path.is_file() or output_path.stat().st_size == 0:
        raise RuntimeError("The generated audio file is empty.")

    return output_path


print("Gemini TTS function is ready.")
print("Model:", GEMINI_TTS_MODEL)
print("Languages: English, Hindi, Marathi")
print("The existing voice-pipeline function signature is preserved.")

Gemini TTS function is ready.
Model: gemini-3.8-flash-lite-tts
Languages: English, Hindi, Marathi
The existing voice-pipeline function signature is preserved.


In [13]:
# Cell 12: Test English, Hindi, and Marathi RAG-to-TTS

from IPython.display import Audio, display

voice_tests = [
    {
        "language": "en",
        "label": "English",
        "question": "What is the fee for Python Programming?",
        "filename": "rag_answer_english.wav",
    },
    {
        "language": "hi",
        "label": "Hindi",
        "question": "Python Programming की फीस कितनी है?",
        "filename": "rag_answer_hindi.wav",
    },
    {
        "language": "mr",
        "label": "Marathi",
        "question": "Python Programming ची फी किती आहे?",
        "filename": "rag_answer_marathi.wav",
    },
]

for test in voice_tests:
    language = test["language"]
    question = test["question"]

    print("\n" + "=" * 65)
    print("LANGUAGE:", test["label"])
    print("STUDENT QUESTION:", question)

    # Retrieve the answer from institute documents.
    result = rag_service.answer(question)

    print("\nAI ANSWER:")
    print(result["answer"])
    print("Grounded:", result["grounded"])
    print("Answer mode:", result["answer_mode"])
    print("Sources:", result["sources"])

    # Do not generate speech for unsupported answers in this test.
    if not result["grounded"]:
        print("Audio skipped because the answer is not grounded.")
        continue

    output_path = TTS_OUTPUT_DIR / test["filename"]

    print("\nGenerating speech...", flush=True)

    output_path = await synthesize_speech(
        text=result["answer"],
        language=language,
        output_path=output_path
    )

    print("Audio saved:", output_path)
    print("File size:", f"{output_path.stat().st_size:,} bytes")

    display(Audio(filename=str(output_path), autoplay=False))

print("\nAll three language tests completed.")


LANGUAGE: English
STUDENT QUESTION: What is the fee for Python Programming?

AI ANSWER:
According to the institute's document, the verified information for Python Programming is: Fee: INR 8000.
Grounded: True
Answer mode: deterministic_course_facts
Sources: [{'source': 'demo_institute_catalog.docx', 'chunk_id': 1, 'similarity': 0.5485}]

Generating speech...
Audio saved: C:\Hehe\AI-Lead-Desk-Runtime-Errors\data\voice_outputs\rag_answer_english.wav
File size: 468,834 bytes



LANGUAGE: Hindi
STUDENT QUESTION: Python Programming की फीस कितनी है?

AI ANSWER:
संस्थान के दस्तावेज़ के अनुसार, Python Programming के लिए उपलब्ध जानकारी: फीस: INR 8000.
Grounded: True
Answer mode: deterministic_course_facts
Sources: [{'source': 'demo_institute_catalog.docx', 'chunk_id': 1, 'similarity': 0.5282}]

Generating speech...
Audio saved: C:\Hehe\AI-Lead-Desk-Runtime-Errors\data\voice_outputs\rag_answer_hindi.wav
File size: 441,954 bytes



LANGUAGE: Marathi
STUDENT QUESTION: Python Programming ची फी किती आहे?

AI ANSWER:
संस्थेच्या दस्तऐवजानुसार, Python Programming साठी उपलब्ध माहिती: फी: INR 8000.
Grounded: True
Answer mode: deterministic_course_facts
Sources: [{'source': 'demo_institute_catalog.docx', 'chunk_id': 1, 'similarity': 0.5211}]

Generating speech...
Audio saved: C:\Hehe\AI-Lead-Desk-Runtime-Errors\data\voice_outputs\rag_answer_marathi.wav
File size: 401,634 bytes



All three language tests completed.


In [17]:
# Cell 13: Complete multilingual voice-to-voice interaction

import re
import time
from pathlib import Path
from IPython.display import Audio, display

# Reuse the current project configuration and services.
VOICE_SAMPLE_DIR = PROJECT_DIR / "data" / "voice_samples"
TTS_OUTPUT_DIR = PROJECT_DIR / "data" / "voice_outputs"
TTS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SUPPORTED_AUDIO_EXTENSIONS = {
    ".wav", ".mp3", ".m4a", ".ogg", ".flac",
    ".mp4", ".mpeg", ".mpga", ".webm"
}

audio_files = sorted(
    [
        p for p in VOICE_SAMPLE_DIR.iterdir()
        if p.is_file()
        and p.suffix.lower() in SUPPORTED_AUDIO_EXTENSIONS
    ],
    key=lambda p: p.stat().st_mtime,
    reverse=True
)

if not audio_files:
    raise FileNotFoundError(
        f"No recordings found in {VOICE_SAMPLE_DIR}"
    )

audio_path = audio_files[0]

print("Recording:", audio_path.name)
print("\n[1/5] Uploading recording...", flush=True)

start = time.perf_counter()

uploaded_audio = gemini_client.files.upload(
    file=str(audio_path)
)

print(
    f"Upload completed in {time.perf_counter() - start:.2f}s",
    flush=True
)

# Identify the original spoken language from the audio itself.
print("\n[2/5] Identifying spoken language...", flush=True)

language_result = gemini_client.interactions.create(
    model="gemini-3.8-flash",
    input=[
        {
            "type": "text",
            "text": (
                "Identify the dominant language actually spoken in this "
                "audio. Choose exactly one: en, hi, or mr. "
                "en means English, hi means Hindi, and mr means Marathi. "
                "Distinguish Hindi from Marathi by listening to the speech, "
                "not merely by identifying the script. If English words "
                "are mixed into Hindi or Marathi, choose the dominant "
                "Indian language. Return only the two-letter code."
            )
        },
        {
            "type": "audio",
            "uri": uploaded_audio.uri,
            "mime_type": uploaded_audio.mime_type
        }
    ]
)

raw_language = (language_result.output_text or "").strip().lower()

if "marathi" in raw_language or re.search(r"\bmr\b", raw_language):
    spoken_language = "mr"
elif "hindi" in raw_language or re.search(r"\bhi\b", raw_language):
    spoken_language = "hi"
elif "english" in raw_language or re.search(r"\ben\b", raw_language):
    spoken_language = "en"
else:
    raise ValueError(
        "Language identification was inconclusive. "
        f"Model returned: {raw_language!r}"
    )

print("Raw language result:", raw_language)
print("Selected language:", spoken_language)

# Transcribe while allowing automatic language detection and code-switching.
print("\n[3/5] Transcribing speech...", flush=True)

start = time.perf_counter()

transcription_result = gemini_client.interactions.create(
    model="gemini-3.5-transcribe",
    input=[
        {
            "type": "audio",
            "uri": uploaded_audio.uri,
            "mime_type": uploaded_audio.mime_type
        }
    ],
    generation_config={
        "transcription_config": {
            "language_codes": [],
            "custom_vocabulary": [
                "Python Programming",
                "Data Science Foundations",
                "Web Development",
                "AI Lead Desk"
            ]
        }
    }
)

transcript = (transcription_result.output_text or "").strip()

if not transcript:
    raise ValueError("The transcription model returned no text.")

print(f"Transcription completed in {time.perf_counter() - start:.2f}s")
print("\nTRANSCRIPT:")
print(transcript)

# Use the detected audio language for the reply templates and LLM prompt.
# This temporary override is for the sequential notebook prototype.
# We'll make it a normal service parameter before multi-user deployment.
import ai_lead_desk.rag_service as rag_module

original_language_detector = rag_module.detect_language

try:
    rag_module.detect_language = lambda _query: spoken_language
    result = rag_service.answer(
    transcript,
    response_language=spoken_language
)
finally:
    rag_module.detect_language = original_language_detector

answer = result["answer"]

print("\n[4/5] AI COUNSELLOR RESPONSE")
print("Answer:", answer)
print("Grounded:", result["grounded"])
print("Answer mode:", result["answer_mode"])
print("Sources:", result["sources"])

# Speak the safe answer, including a fallback if the evidence is missing.
print("\n[5/5] Generating spoken response...", flush=True)

start = time.perf_counter()

spoken_audio_path = await synthesize_speech(
    text=answer,
    language=spoken_language,
    output_path=(
        TTS_OUTPUT_DIR / "complete_voice_response.wav"
    )
)

print(f"TTS completed in {time.perf_counter() - start:.2f}s")
print("Saved audio:", spoken_audio_path)

display(Audio(filename=str(spoken_audio_path), autoplay=False))

print("\nComplete voice interaction finished.")

Recording: Recording (20).m4a

[1/5] Uploading recording...
Upload completed in 3.14s

[2/5] Identifying spoken language...
Raw language result: hi
Selected language: hi

[3/5] Transcribing speech...
Transcription completed in 4.09s

TRANSCRIPT:
Python Programming की प्राइस क्या है?

[4/5] AI COUNSELLOR RESPONSE
Answer: Python Programming कोर्स की फीस **INR 8,000** है।
Grounded: True
Answer mode: model_with_retrieved_evidence
Sources: [{'source': 'demo_institute_catalog.docx', 'chunk_id': 1, 'similarity': 0.5097}]

[5/5] Generating spoken response...
TTS completed in 6.31s
Saved audio: C:\Hehe\AI-Lead-Desk-Runtime-Errors\data\voice_outputs\complete_voice_response.wav



Complete voice interaction finished.


In [20]:
# Cell 15: Create a reusable multilingual voice service

from pathlib import Path

voice_module_path = (
    PROJECT_DIR / "src" / "ai_lead_desk" / "voice_service.py"
)

voice_module_code = r'''
import base64
import re
import time
from pathlib import Path

from google import genai


AUDIO_EXTENSIONS = {
    ".wav", ".mp3", ".m4a", ".ogg", ".flac",
    ".mp4", ".mpeg", ".mpga", ".webm"
}

MAX_AUDIO_BYTES = 25 * 1024 * 1024

LANGUAGE_INSTRUCTIONS = {
    "en": (
        "Speak the supplied text naturally in English. "
        "Do not translate or paraphrase it."
    ),
    "hi": (
        "Speak the supplied text naturally in Hindi. "
        "Do not translate or paraphrase it."
    ),
    "mr": (
        "Speak the supplied text naturally in Marathi. "
        "Do not translate or paraphrase it."
    ),
}


def _normalize_language(raw_language):
    """Normalize the language identifier returned by the audio model."""
    value = (raw_language or "").strip().lower()

    if "marathi" in value or re.search(r"(?<!\w)mr(?!\w)", value):
        return "mr"

    if "hindi" in value or re.search(r"(?<!\w)hi(?!\w)", value):
        return "hi"

    if "english" in value or re.search(r"(?<!\w)en(?!\w)", value):
        return "en"

    return None


def identify_audio_language(gemini_client, uploaded_audio):
    """Identify the dominant spoken language from the original recording."""
    response = gemini_client.interactions.create(
        model="gemini-3.8-flash",
        input=[
            {
                "type": "text",
                "text": (
                    "Identify the dominant language actually spoken in "
                    "this audio. Choose exactly one: en, hi, or mr. "
                    "en means English, hi means Hindi, and mr means "
                    "Marathi. Distinguish Hindi from Marathi by listening "
                    "to the speech, not merely by identifying the script. "
                    "If English words are mixed into Hindi or Marathi, "
                    "choose the dominant language. Return only the code."
                ),
            },
            {
                "type": "audio",
                "uri": uploaded_audio.uri,
                "mime_type": uploaded_audio.mime_type,
            },
        ],
    )

    raw_language = (response.output_text or "").strip()
    language = _normalize_language(raw_language)

    if language is None:
        raise ValueError(
            "Could not confidently identify the spoken language. "
            f"Model response: {raw_language!r}"
        )

    return language


def transcribe_uploaded_audio(gemini_client, uploaded_audio):
    """Transcribe speech without translating it into another language."""
    response = gemini_client.interactions.create(
        model="gemini-3.5-transcribe",
        input=[
            {
                "type": "audio",
                "uri": uploaded_audio.uri,
                "mime_type": uploaded_audio.mime_type,
            }
        ],
        generation_config={
            "transcription_config": {
                "language_codes": [],
                "custom_vocabulary": [
                    "Python Programming",
                    "Data Science Foundations",
                    "Web Development",
                    "AI Lead Desk",
                ],
            }
        },
    )

    transcript = (response.output_text or "").strip()

    if not transcript:
        raise ValueError("The transcription model returned no text.")

    return transcript


def synthesize_speech(
    gemini_client,
    text,
    language,
    output_path,
):
    """Generate audio locally nowhere: this uses the Gemini TTS API."""
    if language not in LANGUAGE_INSTRUCTIONS:
        raise ValueError("Language must be 'en', 'hi', or 'mr'.")

    if not isinstance(text, str) or not text.strip():
        raise ValueError("Text to speak cannot be empty.")

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    response = gemini_client.interactions.create(
        model="gemini-3.8-flash-lite-tts",
        input=[
            {
                "type": "user_input",
                "content": [
                    {
                        "type": "text",
                        "text": text.strip(),
                        "annotations": [
                            {
                                "type": "speech_metadata",
                                "style": LANGUAGE_INSTRUCTIONS[language],
                            }
                        ],
                    }
                ],
            }
        ],
        response_format={"type": "audio"},
        generation_config={
            "speech_config": [{"voice": "Kore"}]
        },
    )

    audio_data = response.output_audio.data

    if not audio_data:
        raise RuntimeError("The TTS model returned no audio.")

    if isinstance(audio_data, str):
        audio_bytes = base64.b64decode(audio_data)
    else:
        audio_bytes = bytes(audio_data)

    output_path.write_bytes(audio_bytes)

    if output_path.stat().st_size == 0:
        raise RuntimeError("The generated audio file is empty.")

    return output_path


def process_voice_message(
    audio_path,
    rag_service,
    gemini_client,
    output_dir,
):
    """
    Process a voice message through:
    upload -> language detection -> transcription ->
    grounded counselling -> speech generation.

    Returns transcript, language, answer, provenance,
    audio path, and stage timings.
    """
    audio_path = Path(audio_path)

    if not audio_path.is_file():
        raise FileNotFoundError(f"Audio file not found: {audio_path}")

    if audio_path.suffix.lower() not in AUDIO_EXTENSIONS:
        raise ValueError(f"Unsupported audio format: {audio_path.suffix}")

    if audio_path.stat().st_size > MAX_AUDIO_BYTES:
        raise ValueError("Audio recording exceeds 25 MB.")

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    timings = {}

    started = time.perf_counter()
    uploaded_audio = gemini_client.files.upload(file=str(audio_path))
    timings["upload_seconds"] = round(
        time.perf_counter() - started, 2
    )

    started = time.perf_counter()
    language = identify_audio_language(gemini_client, uploaded_audio)
    timings["language_detection_seconds"] = round(
        time.perf_counter() - started, 2
    )

    started = time.perf_counter()
    transcript = transcribe_uploaded_audio(gemini_client, uploaded_audio)
    timings["transcription_seconds"] = round(
        time.perf_counter() - started, 2
    )

    started = time.perf_counter()
    result = rag_service.answer(
        transcript,
        response_language=language,
    )
    timings["answer_seconds"] = round(
        time.perf_counter() - started, 2
    )

    output_path = output_dir / f"voice_response_{time.time_ns()}.wav"

    started = time.perf_counter()
    audio_path_out = synthesize_speech(
        gemini_client=gemini_client,
        text=result["answer"],
        language=language,
        output_path=output_path,
    )
    timings["tts_seconds"] = round(
        time.perf_counter() - started, 2
    )

    return {
        "transcript": transcript,
        "language": language,
        "answer": result["answer"],
        "grounded": result["grounded"],
        "answer_mode": result["answer_mode"],
        "sources": result["sources"],
        "audio_path": str(audio_path_out),
        "timings": timings,
    }
'''

voice_module_path.write_text(
    voice_module_code.lstrip(),
    encoding="utf-8",
)

print("Reusable voice service created:")
print(voice_module_path)
print("No API requests were made.")

Reusable voice service created:
C:\Hehe\AI-Lead-Desk-Runtime-Errors\src\ai_lead_desk\voice_service.py
No API requests were made.


In [25]:
# Cell 16: End-to-end voice pipeline test with automatic retries

import sys
import time
import importlib
from IPython.display import Audio, display

SRC_DIR = PROJECT_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import ai_lead_desk.voice_service as voice_service
importlib.reload(voice_service)


# Retry the ORIGINAL Gemini language detector on temporary failures.
def identify_audio_language_with_retry(gemini_client, uploaded_audio):
    max_attempts = 3

    for attempt in range(1, max_attempts + 1):
        try:
            response = gemini_client.interactions.create(
                model="gemini-3.8-flash",
                input=[
                    {
                        "type": "text",
                        "text": (
                            "Identify the dominant language actually spoken "
                            "in this audio. Choose exactly one: en, hi, or mr. "
                            "en means English, hi means Hindi, and mr means "
                            "Marathi. Distinguish Hindi from Marathi by "
                            "listening to the speech, not merely by identifying "
                            "the script. If English words are mixed into Hindi "
                            "or Marathi, choose the dominant language. "
                            "Return only the code."
                        ),
                    },
                    {
                        "type": "audio",
                        "uri": uploaded_audio.uri,
                        "mime_type": uploaded_audio.mime_type,
                    },
                ],
            )

            language = voice_service._normalize_language(
                (response.output_text or "").strip()
            )

            if language not in {"en", "hi", "mr"}:
                raise ValueError(
                    f"Unexpected language detection result: {language!r}"
                )

            return language

        except Exception as exc:
            error_text = (
                f"{type(exc).__name__}: {exc}"
            ).lower()

            temporary_error = any(
                marker in error_text
                for marker in (
                    "503",
                    "service_unavailable",
                    "high demand",
                    "temporarily unavailable",
                    "internalservererror",
                )
            )

            if not temporary_error or attempt == max_attempts:
                raise

            wait_seconds = 2 ** attempt

            print(
                f"Gemini is temporarily unavailable. "
                f"Retrying in {wait_seconds} seconds "
                f"(attempt {attempt + 1}/{max_attempts})..."
            )

            time.sleep(wait_seconds)


# Override language detection for this notebook session only.
# No changes are made to voice_service.py.
voice_service.identify_audio_language = (
    identify_audio_language_with_retry
)

sample_audio = (
    PROJECT_DIR
    / "data"
    / "voice_samples"
    / "Recording (20).m4a"
)

output_dir = PROJECT_DIR / "data" / "voice_outputs"

if not sample_audio.exists():
    print("Recording not found:", sample_audio)

else:
    try:
        print("Processing voice message...")
        print("API requests will be made.\n")

        result = voice_service.process_voice_message(
            audio_path=sample_audio,
            rag_service=rag_service,
            gemini_client=gemini_client,
            output_dir=output_dir,
        )

        print("\nTRANSCRIPT")
        print(result["transcript"])

        print("\nDETECTED LANGUAGE")
        print(result["language"])

        print("\nRAG ANSWER")
        print(result["answer"])

        print("\nGROUNDED IN DOCUMENTS")
        print(result["grounded"])

        print("\nANSWER MODE")
        print(result["answer_mode"])

        print("\nSOURCES")
        print(result["sources"])

        print("\nGENERATED AUDIO")
        print(result["audio_path"])

        print("\nTIMINGS")
        print(result.get("timings", {}))

        display(
            Audio(
                filename=str(result["audio_path"]),
                autoplay=False,
            )
        )

    except Exception as exc:
        print("\nVoice pipeline could not complete.")
        print("Error:", type(exc).__name__, "-", exc)
        print(
            "\nIf Gemini is still returning 503, "
            "the service may need time before it accepts requests again."
        )

Processing voice message...
API requests will be made.


TRANSCRIPT
Python Programming की प्राइस क्या है?

DETECTED LANGUAGE
hi

RAG ANSWER
Python Programming कोर्स की फीस **INR 8,000** है।

GROUNDED IN DOCUMENTS
True

ANSWER MODE
model_with_retrieved_evidence

SOURCES
[{'source': 'demo_institute_catalog.docx', 'chunk_id': 1, 'similarity': 0.5097}]

GENERATED AUDIO
C:\Hehe\AI-Lead-Desk-Runtime-Errors\data\voice_outputs\voice_response_1792176356353300400.wav

TIMINGS
{'upload_seconds': 3.03, 'language_detection_seconds': 24.76, 'transcription_seconds': 4.31, 'answer_seconds': 0.65, 'tts_seconds': 6.35}


In [24]:
# Cell 17: Optimize voice processing by running independent API calls concurrently

module_path = (
    PROJECT_DIR / "src" / "ai_lead_desk" / "voice_service.py"
)

source = module_path.read_text(encoding="utf-8")

old_block = '''    started = time.perf_counter()
    language = identify_audio_language(gemini_client, uploaded_audio)
    timings["language_detection_seconds"] = round(
        time.perf_counter() - started, 2
    )

    started = time.perf_counter()
    transcript = transcribe_uploaded_audio(gemini_client, uploaded_audio)
    timings["transcription_seconds"] = round(
        time.perf_counter() - started, 2
    )
'''

new_block = '''    from concurrent.futures import ThreadPoolExecutor

    def _timed_voice_call(function, *args):
        call_started = time.perf_counter()
        value = function(*args)
        return value, time.perf_counter() - call_started

    # Language detection and transcription both use the same uploaded audio.
    # Run the two independent API requests concurrently.
    with ThreadPoolExecutor(max_workers=2) as executor:
        language_future = executor.submit(
            _timed_voice_call,
            identify_audio_language,
            gemini_client,
            uploaded_audio,
        )

        transcript_future = executor.submit(
            _timed_voice_call,
            transcribe_uploaded_audio,
            gemini_client,
            uploaded_audio,
        )

        language, language_elapsed = language_future.result()
        transcript, transcription_elapsed = transcript_future.result()

    timings["language_detection_seconds"] = round(language_elapsed, 2)
    timings["transcription_seconds"] = round(transcription_elapsed, 2)
'''

if old_block in source:
    updated_source = source.replace(old_block, new_block, 1)
    module_path.write_text(updated_source, encoding="utf-8")
    print("Updated:", module_path)
    print("Language detection and transcription will run concurrently.")
    print("No API requests were made.")

elif new_block in source:
    print("Concurrent processing is already configured.")

else:
    raise RuntimeError(
        "Could not find the expected sequential processing block. "
        "The file was not changed."
    )

Updated: C:\Hehe\AI-Lead-Desk-Runtime-Errors\src\ai_lead_desk\voice_service.py
Language detection and transcription will run concurrently.
No API requests were made.


In [26]:
# Cell 18: Bound language-detection wait time with a fallback

from pathlib import Path

module_path = (
    PROJECT_DIR / "src" / "ai_lead_desk" / "voice_service.py"
)

source = module_path.read_text(encoding="utf-8")

old_block = '''    from concurrent.futures import ThreadPoolExecutor

    def _timed_voice_call(function, *args):
        call_started = time.perf_counter()
        value = function(*args)
        return value, time.perf_counter() - call_started

    # Language detection and transcription both use the same uploaded audio.
    # Run the two independent API requests concurrently.
    with ThreadPoolExecutor(max_workers=2) as executor:
        language_future = executor.submit(
            _timed_voice_call,
            identify_audio_language,
            gemini_client,
            uploaded_audio,
        )

        transcript_future = executor.submit(
            _timed_voice_call,
            transcribe_uploaded_audio,
            gemini_client,
            uploaded_audio,
        )

        language, language_elapsed = language_future.result()
        transcript, transcription_elapsed = transcript_future.result()

    timings["language_detection_seconds"] = round(language_elapsed, 2)
    timings["transcription_seconds"] = round(transcription_elapsed, 2)
'''

new_block = '''    from concurrent.futures import ThreadPoolExecutor, TimeoutError

    def _timed_voice_call(function, *args):
        call_started = time.perf_counter()
        value = function(*args)
        return value, time.perf_counter() - call_started

    def _fallback_language(transcript):
        # Best-effort fallback. Audio-based Gemini detection remains preferred.
        text = transcript.lower()

        marathi_markers = [
            "आहे", "आहेत", "काय", "किती", "मला",
            "माझा", "माझी", "माझे", "मध्ये", "साठी",
            "सांगा", "पाहिजे", "aahe", "ahet", "kay",
            "kiti", "mala", "majha", "majhi", "madhe",
            "sathi", "sanga", "pahije",
        ]

        hindi_markers = [
            "क्या", "है", "हैं", "मुझे", "मेरा",
            "मेरी", "में", "कितना", "कितनी", "बताओ",
            "चाहिए", "कहाँ", "kya", "hai", "hain",
            "mujhe", "mera", "meri", "mein", "kitna",
            "kitni", "batao", "chahiye", "kahan",
        ]

        mr_score = sum(text.count(word) for word in marathi_markers)
        hi_score = sum(text.count(word) for word in hindi_markers)

        if mr_score > hi_score and mr_score > 0:
            return "mr"

        if hi_score > mr_score and hi_score > 0:
            return "hi"

        if re.search(r"[\\\\u0900-\\\\u097F]", text):
            return "hi"

        return "en"

    executor = ThreadPoolExecutor(max_workers=2)

    try:
        language_future = executor.submit(
            _timed_voice_call,
            identify_audio_language,
            gemini_client,
            uploaded_audio,
        )

        transcript_future = executor.submit(
            _timed_voice_call,
            transcribe_uploaded_audio,
            gemini_client,
            uploaded_audio,
        )

        # Obtain the transcript while language detection runs in parallel.
        transcript, transcription_elapsed = transcript_future.result()
        timings["transcription_seconds"] = round(
            transcription_elapsed, 2
        )

        language_wait_started = time.perf_counter()

        try:
            language, language_elapsed = language_future.result(
                timeout=8
            )
            timings["language_detection_seconds"] = round(
                language_elapsed, 2
            )
            timings["language_detection_method"] = "audio"

        except (TimeoutError, Exception) as exc:
            language = _fallback_language(transcript)
            timings["language_detection_seconds"] = round(
                time.perf_counter() - language_wait_started, 2
            )
            timings["language_detection_method"] = "transcript_fallback"

            print(
                "Warning: audio-based language detection did not finish "
                f"successfully ({type(exc).__name__}). "
                f"Using best-effort transcript fallback: {language}"
            )

    finally:
        # Don't make the user wait for a slow detector after fallback.
        executor.shutdown(wait=False, cancel_futures=True)
'''

if old_block in source:
    updated_source = source.replace(old_block, new_block, 1)
    module_path.write_text(updated_source, encoding="utf-8")

    # Compile-check the modified file without importing or making API calls.
    compile(updated_source, str(module_path), "exec")

    print("Updated:", module_path)
    print("Original audio detector preserved as the preferred path.")
    print("Fallback enabled after an 8-second wait.")
    print("No API requests were made.")

elif new_block in source:
    print("Timeout-based language fallback is already configured.")

else:
    raise RuntimeError(
        "Could not locate the expected concurrent-processing block. "
        "The file was not changed."
    )

Updated: C:\Hehe\AI-Lead-Desk-Runtime-Errors\src\ai_lead_desk\voice_service.py
Original audio detector preserved as the preferred path.
Fallback enabled after an 8-second wait.
No API requests were made.


In [27]:
# Cell 16: End-to-end voice pipeline test with automatic retries

import sys
import time
import importlib
from IPython.display import Audio, display

SRC_DIR = PROJECT_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import ai_lead_desk.voice_service as voice_service
importlib.reload(voice_service)


# Retry the ORIGINAL Gemini language detector on temporary failures.
def identify_audio_language_with_retry(gemini_client, uploaded_audio):
    max_attempts = 3

    for attempt in range(1, max_attempts + 1):
        try:
            response = gemini_client.interactions.create(
                model="gemini-3.8-flash",
                input=[
                    {
                        "type": "text",
                        "text": (
                            "Identify the dominant language actually spoken "
                            "in this audio. Choose exactly one: en, hi, or mr. "
                            "en means English, hi means Hindi, and mr means "
                            "Marathi. Distinguish Hindi from Marathi by "
                            "listening to the speech, not merely by identifying "
                            "the script. If English words are mixed into Hindi "
                            "or Marathi, choose the dominant language. "
                            "Return only the code."
                        ),
                    },
                    {
                        "type": "audio",
                        "uri": uploaded_audio.uri,
                        "mime_type": uploaded_audio.mime_type,
                    },
                ],
            )

            language = voice_service._normalize_language(
                (response.output_text or "").strip()
            )

            if language not in {"en", "hi", "mr"}:
                raise ValueError(
                    f"Unexpected language detection result: {language!r}"
                )

            return language

        except Exception as exc:
            error_text = (
                f"{type(exc).__name__}: {exc}"
            ).lower()

            temporary_error = any(
                marker in error_text
                for marker in (
                    "503",
                    "service_unavailable",
                    "high demand",
                    "temporarily unavailable",
                    "internalservererror",
                )
            )

            if not temporary_error or attempt == max_attempts:
                raise

            wait_seconds = 2 ** attempt

            print(
                f"Gemini is temporarily unavailable. "
                f"Retrying in {wait_seconds} seconds "
                f"(attempt {attempt + 1}/{max_attempts})..."
            )

            time.sleep(wait_seconds)


# Override language detection for this notebook session only.
# No changes are made to voice_service.py.
voice_service.identify_audio_language = (
    identify_audio_language_with_retry
)

sample_audio = (
    PROJECT_DIR
    / "data"
    / "voice_samples"
    / "Recording (20).m4a"
)

output_dir = PROJECT_DIR / "data" / "voice_outputs"

if not sample_audio.exists():
    print("Recording not found:", sample_audio)

else:
    try:
        print("Processing voice message...")
        print("API requests will be made.\n")

        result = voice_service.process_voice_message(
            audio_path=sample_audio,
            rag_service=rag_service,
            gemini_client=gemini_client,
            output_dir=output_dir,
        )

        print("\nTRANSCRIPT")
        print(result["transcript"])

        print("\nDETECTED LANGUAGE")
        print(result["language"])

        print("\nRAG ANSWER")
        print(result["answer"])

        print("\nGROUNDED IN DOCUMENTS")
        print(result["grounded"])

        print("\nANSWER MODE")
        print(result["answer_mode"])

        print("\nSOURCES")
        print(result["sources"])

        print("\nGENERATED AUDIO")
        print(result["audio_path"])

        print("\nTIMINGS")
        print(result.get("timings", {}))

        display(
            Audio(
                filename=str(result["audio_path"]),
                autoplay=False,
            )
        )

    except Exception as exc:
        print("\nVoice pipeline could not complete.")
        print("Error:", type(exc).__name__, "-", exc)
        print(
            "\nIf Gemini is still returning 503, "
            "the service may need time before it accepts requests again."
        )

Processing voice message...
API requests will be made.


TRANSCRIPT
Python Programming की प्राइस क्या है?

DETECTED LANGUAGE
hi

RAG ANSWER
Python Programming कोर्स की फीस **INR 8,000** है।

GROUNDED IN DOCUMENTS
True

ANSWER MODE
model_with_retrieved_evidence

SOURCES
[{'source': 'demo_institute_catalog.docx', 'chunk_id': 1, 'similarity': 0.5097}]

GENERATED AUDIO
C:\Hehe\AI-Lead-Desk-Runtime-Errors\data\voice_outputs\voice_response_1792176552985923800.wav

TIMINGS
{'upload_seconds': 3.15, 'transcription_seconds': 4.12, 'language_detection_seconds': 8.0, 'language_detection_method': 'transcript_fallback', 'answer_seconds': 0.64, 'tts_seconds': 5.32}


In [28]:
# Cell 19: Inspect the existing RAG service interface
# This is an inspection only. No API requests are made.

import sys
import inspect

SRC_DIR = PROJECT_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ai_lead_desk.rag_service import DocumentRAGService

print("RAG SERVICE CLASS")
print(DocumentRAGService)

print("\nCONSTRUCTOR SIGNATURE")
print(inspect.signature(DocumentRAGService))

print("\nEXISTING INSTANCE TYPE")
print(type(rag_service))

print("\nANSWER METHOD SIGNATURE")
print(inspect.signature(rag_service.answer))

print("\nCONSTRUCTOR IMPLEMENTATION")
print(inspect.getsource(DocumentRAGService.__init__))

print("\nANSWER METHOD IMPLEMENTATION")
print(inspect.getsource(DocumentRAGService.answer))

print("\nPUBLIC METHODS")
print([
    name
    for name, member in inspect.getmembers(
        DocumentRAGService,
        predicate=inspect.isfunction,
    )
    if not name.startswith("_")
])

print("\nInspection complete. No API requests were made.")

RAG SERVICE CLASS
<class 'ai_lead_desk.rag_service.DocumentRAGService'>

CONSTRUCTOR SIGNATURE
(project_dir, embedding_model_name='nomic-embed-text-v2-moe:latest', llm_model_name='openai/gpt-oss-120b')

EXISTING INSTANCE TYPE
<class 'ai_lead_desk.rag_service.DocumentRAGService'>

ANSWER METHOD SIGNATURE
(query, response_language=None)

CONSTRUCTOR IMPLEMENTATION
    def __init__(
        self,
        project_dir,
        embedding_model_name="nomic-embed-text-v2-moe:latest",
        llm_model_name="openai/gpt-oss-120b"
    ):
        self.project_dir = Path(project_dir).resolve()
        self.documents_dir = (
            self.project_dir / "data" / "institute_documents"
        )

        load_dotenv(self.project_dir / ".env")

        if not os.getenv("GROQ_API_KEY"):
            raise EnvironmentError(
                "GROQ_API_KEY is missing from the project .env file."
            )

        self.llm = ChatGroq(
            model=llm_model_name,
            temperature=0,
       